# Fake News Generator & Detection
Use of Generative AI to create and detect fake news using GPT-2 and BERT.

###  Setup & Install Required Libraries

In [1]:
!pip install newspaper3k lxml_html_clean pandas transformers torch gradio


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 38.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 3.8 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=071ae1bc14f49b10a4ede6d74e1f73613a827f7cfaf953e6d2739610343e34ab
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=e07b384fac6d0a8746886ecaec986c09689a7d825271e4724ac521e38e70c04f
  Stored in directory: /root/.cache/pip/wheels/9f/9f/fb/364871d7426d3cdd4d293dcf7e53d97f160

##  Import Libraries

In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import gradio as gr


In [3]:
pip install lxml_html_clean

In [4]:
import lxml_html_clean

In [5]:
from newspaper import Article

## Fake News Generator using GPT-2 & BERT

In [9]:
# Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
# DATASET & TRAINING
df = pd.DataFrame({
    'text': [
        # REAL NEWS
        "The central bank today announced a slight increase in interest rates to combat rising inflation.",
        "Local government officials have approved the budget for a new public transportation initiative.",
        "A recent clinical study suggests that a balanced diet significantly improves long-term heart health.",
        "The international summit concluded with a joint statement on global trade and economic cooperation.",
        "Energy experts report a 15% increase in the use of renewable power sources over the last year.",
        # FAKE NEWS
        "SHOCKING: Scientists discover that humans can now breathe underwater using a new secret pill!",
        "LEAKED: The government is planning to replace all physical money with chocolate by next Monday.",
        "MUST WATCH: A giant dragon was spotted flying over the city center yesterday evening.",
        "Doctors are furious! This one simple trick with a lemon can cure every known disease instantly.",
        "URGENT: NASA confirms that a fleet of alien spaceships is currently hiding behind the moon."
    ],  'label': [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
})
train_texts, val_texts, train_labels, val_labels = train_test_split(df['text'], df['label'], test_size=0.2)
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = bert_tokenizer(list(train_texts), truncation=True, padding=True, return_tensors="pt")
val_encodings = bert_tokenizer(list(val_texts), truncation=True, padding=True, return_tensors="pt")

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, list(train_labels))
val_dataset = NewsDataset(val_encodings, list(val_labels))

bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2).to(device)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_dir='./logs',
    report_to="none"
)
trainer = Trainer(model=bert_model, args=training_args, train_dataset=train_dataset)
trainer.train()

#GENERATOR
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

#URL Scraping
def scrape_url(url):
    if not url.strip():
        return "Please enter a valid URL."
    try:
        article = Article(url)
        article.download()
        article.parse()
        if not article.text.strip():
            return "Failed to extract text. This site might be blocking scrapers. Try a different news source."
        return article.text
    except Exception as e:
        return f"Error fetching article: {str(e)}"

def generate_fake_news(prompt):
    inputs = gpt2_tokenizer.encode(prompt, return_tensors="pt").to(device)
    outputs = gpt2_model.generate(
        inputs,
        max_length=200,
        num_return_sequences=1,
        do_sample=True,
        temperature=0.9,
        top_k=50,
        top_p=0.95,
        early_stopping=True,
        pad_token_id=gpt2_tokenizer.eos_token_id
    )
    return gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)

def detect_news(text):
    #Clean the text(removes extra whitespace)
    text = " ".join(text.split()) # BERT max limit is 512 tokens
    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)
    bert_model.eval()
    with torch.no_grad():
        outputs = bert_model(**inputs)
    #Probabilities
    probs = torch.softmax(outputs.logits, dim=1)
    real_prob = probs[0][0].item()
    fake_prob = probs[0][1].item()
    predicted_class = torch.argmax(probs, dim=1).item()
    confidence = probs[0][predicted_class].item()
    label = "Fake News" if predicted_class == 1 else "Real News"
    return f"RESULT: {label} | Confidence: {confidence:.2f}"

# Gradio UI
with gr.Blocks(theme=gr.themes.Glass()) as demo:
    gr.Markdown("# Fake News: GPT-2 Generator & BERT Detector")
    with gr.Row():
        # Column 1: Generator
        with gr.Column(variant="panel"):
            gr.Markdown(" Generate News")
            gen_input = gr.Textbox(label="Prompt", placeholder="Enter your prompt...", lines=2)
            gen_btn = gr.Button("Generate", variant="primary")
            gen_output = gr.Textbox(label="Generated fake news", lines=6)
            gen_btn.click(generate_fake_news, inputs=gen_input, outputs=gen_output)

        # Column 2: Detector with URL Feature
        with gr.Column(variant="panel"):
            gr.Markdown(" Detect Truth")
            # section for URL
            url_input = gr.Textbox(label="Scrape from URL", placeholder="Paste news link here...")
            scrape_btn = gr.Button("Extract Text from URL")
            det_input = gr.Textbox(label="Paste News Text (or Extracted Text)", placeholder="Paste text here...", lines=5)
            scrape_btn.click(scrape_url, inputs=url_input, outputs=det_input)

            det_btn = gr.Button("Detect", variant="secondary")
            det_output = gr.Textbox(label="Result")
            det_btn.click(detect_news, inputs=det_input, outputs=det_output)
if __name__ == "__main__":
    demo.launch()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipython-input-675/4122333554.py:106: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Glass()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0486e67f8d9f632c6f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
